In [1]:
# =========================
# Cell 1 — Setup
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# ---- Set your Kaggle dataset folder here ----
# It should contain the raw CSVs, and the output from dataset_test_preprocessing.ipynb:
#   processed_clean/ashrae_test_merged_dataset/
DATA_DIR = Path(r"E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction")
assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR}"

IN_DIR = DATA_DIR / "processed_clean" / "ashrae_test_merged_dataset"  # partitioned parquet dir from test preprocessing
OUT_DIR = DATA_DIR / "processed_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_DATASET_DIR = OUT_DIR / "ashrae_test_merged_plus_manual_features"

print("Input :", IN_DIR.resolve())
print("Output:", OUT_DATASET_DIR.resolve())

Input : E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction\processed_clean\ashrae_test_merged_dataset
Output: E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction\processed_features\ashrae_test_merged_plus_manual_features


In [2]:
# =========================
# Cell 2 — Load cleaned test parquet dataset
# =========================
# Partitioned parquet dataset loads fine with pandas if pyarrow is set up.
df = pd.read_parquet(IN_DIR, engine="pyarrow")
print("Loaded shape:", df.shape)
display(df.head(3))

Loaded shape: (41697600, 20)


,row_id,building_id,timestamp,ts_idx,pair_id,primary_use,square_feet,year_built,floor_count,timestamp_gmt,time_diff_hours,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed,site_id,meter
0,0,0,2017-01-01,8784,0,Education,7432,2008.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6,0,0
1,1,1,2017-01-01,8784,4,Education,2720,2004.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6,0,0
2,2,2,2017-01-01,8784,8,Education,5376,1991.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6,0,0


In [3]:
# =========================
# Cell 3 — Basic sanity checks (required columns)
# =========================
# NOTE: test has NO `meter_reading`, so do not require it here.
required_cols = [
    "row_id",
    "timestamp",
    "site_id",
    "building_id",
    "meter",
    "air_temperature",
    "dew_temperature",
    "square_feet",
    "year_built",
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing required cols: {missing}"

assert pd.api.types.is_datetime64_any_dtype(df["timestamp"]), "timestamp must be datetime64"
assert len(df) == df["row_id"].nunique(), "row_id must remain unique (no drops/duplication)."

print("timestamp range:", df["timestamp"].min(), "->", df["timestamp"].max())
print("Null air_temperature:", int(df["air_temperature"].isna().sum()))
print("Null square_feet:", int(df["square_feet"].isna().sum()))
print("Null year_built:", int(df["year_built"].isna().sum()))

timestamp range: 2017-01-01 00:00:00 -> 2018-12-31 23:00:00
Null air_temperature: 0
Null square_feet: 0
Null year_built: 24598080


In [4]:
# =========================
# Cell 4 — Holiday features: North America vs Western Europe (UK+IE)
# =========================
# Requires: pip install holidays
import holidays
import pandas as pd

years = sorted(df["timestamp"].dt.year.unique().tolist())
dates = df["timestamp"].dt.date  # python date objects, good for holidays lib

# North America = US + Canada (federal holidays)
us_hols = holidays.US(years=years)
ca_hols = holidays.Canada(years=years)

# Western Europe (for GEPIII sites this is primarily UK + Ireland)
# UnitedKingdom includes bank holidays (incl. May Day / Spring bank holiday, etc.)
uk_hols = holidays.UnitedKingdom(years=years)
ie_hols = holidays.Ireland(years=years)

na_holidays = us_hols | ca_hols
eu_holidays = uk_hols | ie_hols

# Vectorized membership flags
df["is_na_holiday"] = pd.Series(dates).isin(na_holidays).astype("int8").to_numpy()
df["is_eu_holiday"] = pd.Series(dates).isin(eu_holidays).astype("int8").to_numpy()

print("Holiday feature means:")
print("is_na_holiday:", float(df["is_na_holiday"].mean()))
print("is_eu_holiday:", float(df["is_eu_holiday"].mean()))

Holiday feature means:
is_na_holiday: 0.036986301369863014
is_eu_holiday: 0.031506849315068496


In [5]:
# =========================
# Cell 5 — Feature engineering (uses NA/EU holidays)
# Implements: hour/day/season cycles, weekend, holiday, business hours,
#            CDH/HDH, temp regime flags, dewpoint depression, log_sqft, year_built clipping
# Assumes Cell 4 created: is_na_holiday, is_eu_holiday
# =========================

import numpy as np
import pandas as pd

# --- Timestamp components ---
ts = df["timestamp"]
hour = ts.dt.hour.astype("int8")
dow = ts.dt.dayofweek.astype("int8")  # Monday=0 ... Sunday=6
doy = ts.dt.dayofyear.astype("int16")  # 1..366

# (1) hour cyclic
df["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype("float32")
df["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype("float32")

# (2) day-of-week (kept as integer; encoding later)
df["dayofweek"] = dow

# (3) seasonal cycle via day-of-year cyclic
# Use 365.25 to keep leap-year continuity smooth.
df["doy_sin"] = np.sin(2 * np.pi * (doy / 365.25)).astype("float32")
df["doy_cos"] = np.cos(2 * np.pi * (doy / 365.25)).astype("float32")

# (4) weekend flag
df["is_weekend"] = (dow >= 5).astype("int8")

# (5) holiday flags already created in Cell 4:
assert (
    "is_na_holiday" in df.columns and "is_eu_holiday" in df.columns
), "Run Cell 4 first to create is_na_holiday and is_eu_holiday."

# Optional: an "any holiday" helper (useful for business-hours rule)
df["is_holiday_any"] = ((df["is_na_holiday"] == 1) | (df["is_eu_holiday"] == 1)).astype("int8")

# (6) business-hours indicator (weekday + 8–18 + not holiday)
df["is_business_hours"] = ((df["is_weekend"] == 0) & (df["is_holiday_any"] == 0) & (hour >= 8) & (hour <= 18)).astype(
    "int8"
)

# (7) CDH/HDH with fixed base temperature (18C)
T_base = 18.0
T = df["air_temperature"].astype("float32")
df["CDH_18C"] = np.maximum(0.0, T - T_base).astype("float32")
df["HDH_18C"] = np.maximum(0.0, T_base - T).astype("float32")

# (8) temperature regime flags with fixed thresholds
df["is_hot_24C"] = (T > 24.0).astype("int8")
df["is_cold_10C"] = (T < 10.0).astype("int8")

# (9) dewpoint depression (humidity proxy)
df["dewpoint_depression"] = (df["air_temperature"] - df["dew_temperature"]).astype("float32")

# (12) log building area
df["log_sqft"] = np.log1p(df["square_feet"].astype("float32")).astype("float32")

# (13) transform year_built only (clipped to valid range; keep NaN if missing)
yb = df["year_built"].astype("float32")
yb = yb.where(yb.isna() | ((yb >= 1900) & (yb <= 2018)), np.nan)
df["year_built_clipped"] = yb

# --- Report engineered columns created in this cell ---
engineered_cols = [
    "hour_sin",
    "hour_cos",
    "dayofweek",
    "doy_sin",
    "doy_cos",
    "is_weekend",
    "is_holiday_any",
    "is_business_hours",
    "CDH_18C",
    "HDH_18C",
    "is_hot_24C",
    "is_cold_10C",
    "dewpoint_depression",
    "log_sqft",
    "year_built_clipped",
]
print("Engineered columns created/updated in Cell 5:")
print(engineered_cols)

Engineered columns created/updated in Cell 5:
['hour_sin', 'hour_cos', 'dayofweek', 'doy_sin', 'doy_cos', 'is_weekend', 'is_holiday_any', 'is_business_hours', 'CDH_18C', 'HDH_18C', 'is_hot_24C', 'is_cold_10C', 'dewpoint_depression', 'log_sqft', 'year_built_clipped']


In [6]:
# =========================
# Cell 6 — Sanity checks for engineered features
# =========================
import numpy as np
import pandas as pd

engineered_cols = [
    "hour_sin",
    "hour_cos",
    "dayofweek",
    "doy_sin",
    "doy_cos",
    "is_weekend",
    "is_na_holiday",
    "is_eu_holiday",
    "is_holiday_any",
    "is_business_hours",
    "CDH_18C",
    "HDH_18C",
    "is_hot_24C",
    "is_cold_10C",
    "dewpoint_depression",
    "log_sqft",
    "year_built_clipped",
]

missing_cols = [c for c in engineered_cols if c not in df.columns]
assert not missing_cols, f"Missing engineered cols: {missing_cols}"

checks = {
    "hour_sin_range": (float(df["hour_sin"].min()), float(df["hour_sin"].max())),
    "hour_cos_range": (float(df["hour_cos"].min()), float(df["hour_cos"].max())),
    "dayofweek_unique": sorted(df["dayofweek"].unique().tolist()),
    "doy_sin_range": (float(df["doy_sin"].min()), float(df["doy_sin"].max())),
    "doy_cos_range": (float(df["doy_cos"].min()), float(df["doy_cos"].max())),
    "is_weekend_mean": float(df["is_weekend"].mean()),
    "is_na_holiday_mean": float(df["is_na_holiday"].mean()),
    "is_eu_holiday_mean": float(df["is_eu_holiday"].mean()),
    "is_holiday_any_mean": float(df["is_holiday_any"].mean()),
    "holiday_overlap_mean": float(((df["is_na_holiday"] == 1) & (df["is_eu_holiday"] == 1)).mean()),
    "is_business_hours_mean": float(df["is_business_hours"].mean()),
    "CDH_18C_mean": float(df["CDH_18C"].mean()),
    "HDH_18C_mean": float(df["HDH_18C"].mean()),
    "is_hot_24C_mean": float(df["is_hot_24C"].mean()),
    "is_cold_10C_mean": float(df["is_cold_10C"].mean()),
    "dewpoint_depression_mean": float(df["dewpoint_depression"].mean()),
    "log_sqft_range": (float(df["log_sqft"].min()), float(df["log_sqft"].max())),
    "year_built_clipped_missing_frac": float(df["year_built_clipped"].isna().mean()),
    "year_built_clipped_range": (
        float(df["year_built_clipped"].min(skipna=True)),
        float(df["year_built_clipped"].max(skipna=True)),
    ),
}

for k, v in checks.items():
    print(f"{k}: {v}")

# Show a small sample
display(df[["row_id"] + engineered_cols].head(5))

print("\nHoliday any vs weekend (counts):")
display(pd.crosstab(df["is_holiday_any"], df["is_weekend"]))

print("\nBusiness hours vs weekend (counts):")
display(pd.crosstab(df["is_business_hours"], df["is_weekend"]))

hour_sin_range: (-1.0, 1.0)
hour_cos_range: (-1.0, 1.0)
dayofweek_unique: [0, 1, 2, 3, 4, 5, 6]
doy_sin_range: (-0.9999994039535522, 0.9999855756759644)
doy_cos_range: (-0.9999791979789734, 0.9999907612800598)
is_weekend_mean: 0.2863013698630137
is_na_holiday_mean: 0.036986301369863014
is_eu_holiday_mean: 0.031506849315068496
is_holiday_any_mean: 0.056164383561643834
holiday_overlap_mean: 0.012328767123287671
is_business_hours_mean: 0.3051369863013699
CDH_18C_mean: 3.3514411449432373
HDH_18C_mean: 5.8837456703186035
is_hot_24C_mean: 0.23867558324699742
is_cold_10C_mean: 0.2997340854149879
dewpoint_depression_mean: 7.901149749755859
log_sqft_range: (5.648974418640137, 13.68198013305664)
year_built_clipped_missing_frac: 0.5899159663865546
year_built_clipped_range: (1900.0, 2017.0)


,row_id,hour_sin,hour_cos,dayofweek,doy_sin,doy_cos,is_weekend,is_na_holiday,is_eu_holiday,is_holiday_any,is_business_hours,CDH_18C,HDH_18C,is_hot_24C,is_cold_10C,dewpoint_depression,log_sqft,year_built_clipped
0,0,0.0,1.0,6,0.017202,0.999852,1,1,1,1,0,0.0,1.299999,0,0,3.400001,8.913685,2008.0
1,1,0.0,1.0,6,0.017202,0.999852,1,1,1,1,0,0.0,1.299999,0,0,3.400001,7.908755,2004.0
2,2,0.0,1.0,6,0.017202,0.999852,1,1,1,1,0,0.0,1.299999,0,0,3.400001,8.589886,1991.0
3,3,0.0,1.0,6,0.017202,0.999852,1,1,1,1,0,0.0,1.299999,0,0,3.400001,10.072639,2002.0
4,4,0.0,1.0,6,0.017202,0.999852,1,1,1,1,0,0.0,1.299999,0,0,3.400001,11.666574,1975.0



Holiday any vs weekend (counts):


is_weekend,0,1
is_holiday_any,,
0,27760320,11595360
1,1999200,342720



Business hours vs weekend (counts):


is_weekend,0,1
is_business_hours,,
0,17036040,11938080
1,12723480,0


In [7]:
# =========================
# Cell 7 — Save as a new parquet dataset (do not overwrite original)
# =========================
# Save partitioned by site_id and meter again to keep file sizes manageable.
# Keep row_id for later submission alignment.

if OUT_DATASET_DIR.exists():
    # safety: avoid accidental overwrite
    raise FileExistsError(f"Output directory already exists: {OUT_DATASET_DIR}")

OUT_DATASET_DIR.mkdir(parents=True, exist_ok=False)

df.to_parquet(
    OUT_DATASET_DIR,
    index=False,
    engine="pyarrow",
    compression="snappy",
    partition_cols=["site_id", "meter"],
)

print("Saved engineered TEST dataset to:", OUT_DATASET_DIR.resolve())
print("Final shape:", df.shape)
print("Unique row_id:", df["row_id"].nunique())

Saved engineered TEST dataset to: E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction\processed_features\ashrae_test_merged_plus_manual_features
Final shape: (41697600, 37)
Unique row_id: 41697600
